In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# ======================
# Dtypes optimisés
# ======================
dtype_map = {
    "ID": "string",
    "Severity": "int8",
    "Start_Lat": "float32",
    "Start_Lng": "float32",
    "End_Lat": "float32",
    "End_Lng": "float32",
    "Distance(mi)": "float32",
    "City": "category",
    "County": "category",
    "State": "category",
    "Wind_Direction": "category",
    "Weather_Condition": "category",
    "Sunrise_Sunset": "category",
    "Civil_Twilight": "category",
}

bool_cols = [
    "Amenity","Bump","Crossing","Give_Way","Junction","No_Exit",
    "Railway","Roundabout","Station","Stop","Traffic_Calming","Traffic_Signal"
]

drop_cols = [
    "Country", "Source", "Turning_Loop", "Description", "Street",
    "Zipcode", "Timezone", "Nautical_Twilight", "Astronomical_Twilight"
]

In [ ]:
df = pd.read_csv(
    "us.csv",
    dtype=dtype_map,
    parse_dates=["Start_Time", "End_Time", "Weather_Timestamp"]
)

In [ ]:
for col in bool_cols:
    df[col] = df[col].astype("boolean")

In [ ]:
df.drop(columns=drop_cols, errors="ignore", inplace=True)
df.drop(columns=["End_Lat", "End_Lng"], errors="ignore", inplace=True)

In [ ]:
df["Temperature(C)"] = (df["Temperature(F)"] - 32) * 5/9
df["Wind_Chill(C)"] = (df["Wind_Chill(F)"] - 32) * 5/9
df.drop(columns=["Temperature(F)", "Wind_Chill(F)"], errors="ignore", inplace=True)

In [ ]:
df = df.sort_values(["Airport_Code", "Start_Time"])
df["Weather_Timestamp"] = df.groupby("Airport_Code")["Weather_Timestamp"].ffill()

In [ ]:
cols_to_impute = ["Temperature(C)", "Humidity(%)", "Pressure(in)",
                  "Visibility(mi)", "Wind_Speed(mph)"]

df["Wind_Chill(C)"] = df["Wind_Chill(C)"].fillna(0)
df["Precipitation(in)"] = df["Precipitation(in)"].fillna(0)

imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=30, max_depth=6, n_jobs=-1),
    max_iter=5,
    random_state=42
)

df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])


In [ ]:
for c in cols_to_impute + ["Wind_Chill(C)", "Precipitation(in)"]:
    df[c] = df[c].astype("float32")

print("✅ Nettoyage + Imputation COMPLET effectué.")

In [ ]:
# Résumé des colonnes
pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.values,
    "Missing Values": df.isna().sum().values
})

In [ ]:
# Colonnes catégorielles
cat_cols = ["City", "Airport_Code", "Wind_Direction", "Weather_Condition",
            "Sunrise_Sunset", "Civil_Twilight"]

for col in cat_cols:
    df[col] = df[col].astype("category")
    if "Unknown" not in df[col].cat.categories:
        df[col] = df[col].cat.add_categories("Unknown")
    df[col] = df[col].fillna("Unknown")

print("✅ Colonnes catégorielles nettoyées et valeurs manquantes remplacées par 'Unknown'.")

# -------------------------------
# 2️⃣ Forward fill pour Weather_Timestamp
# -------------------------------
# On trie par Airport_Code puis Start_Time pour que le forward fill ait du sens
df = df.sort_values(["Airport_Code", "Start_Time"])

# Remplir les timestamps manquants avec la dernière valeur connue de la même station
df["Weather_Timestamp"] = df.groupby("Airport_Code")["Weather_Timestamp"].ffill()

# Vérification rapide
print("✅ Forward fill effectué pour Weather_Timestamp.")
print(df[["Airport_Code", "Start_Time", "Weather_Timestamp"]].head(10))
